# MT Hexapod current noise at faults

This notebook will search timestamps for M2Hex faults
and analyze the current noise when it faults.\
We will only consider Faults with error codes equal to 1\
For a robust analysis, this notebook anaylzes since the start of observations in April

Noise being the standard deviation of the current in the motors, given by the EfdClient

M2Hex has several faults associated with motor oscillations. We want to characterize these oscillations/vibrations.\
I suggest the creation of a function that applies FFT to a motor current and extracts the most dominant frequency components.\
I will leave the technical aspects of the implementation open for now since there might be some pre-processing we need to apply to the data.



## General Data

First, lest stablish some general variables.

We'll start querying the last month, then we can change to what we see fit.\
Another important variable is the time window where the fault(s) happens.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from astropy.time import Time, TimeDelta
from scipy.fft import fft, ifft
from scipy.signal import detrend

from lsst.summit.utils.efdUtils import getEfdData, getDayObsEndTime, getDayObsStartTime, makeEfdClient

In [ ]:
# Strut pairs     1    6    2    4    3    5
# motorCurrent    0    5    1    3    2    4

day_start = 20250630
day_end = 20250731


N_STRUTS = 6

error_code = 1 # Will only consider errorCode 1 as Faults
sal_index = 2 # M2Hexapods (maybe make it to search for CamHex faults aswell?)

# Upper and lower time difference from the fault
delta_start = 45
delta_end = 5

# Verbose to see what the notebook is doing (don't understand logger yet)
verbose = True

# Pandas configuration
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Query the timestamps

We need to know when do these faults happened

In [ ]:
# Create the client for InfluxQL
efd_client = makeEfdClient()

In [ ]:
start_time = getDayObsStartTime(day_start)
end_time = getDayObsEndTime(day_end)

# Query Hexapod faults
df_timestamps = getEfdData(
        client=efd_client,
        topic='lsst.sal.MTHexapod.logevent_errorCode',
        columns=["errorCode", "errorReport", "salIndex", "traceback"],
        begin=start_time,
        end=end_time,
    )

df_timestamps = df_timestamps[df_timestamps.salIndex == sal_index]

# Know which Timestamps caused errorCode = 1 Faults
timestamps_error_code = df_timestamps[df_timestamps['errorCode'] == error_code].index

if verbose == True:
    print(timestamps_error_code)

In [ ]:
df_timestamps

## Analysis of Current Noise

Now that we have the timestamps, we can query these timestamps and analyze the noise of the current in the proximity of these faults.

In [ ]:
def fault_current_noise_query(motor, delta_start, delta_end, timestamp):
    '''
    Query the noise current for a certain motor in a timelapse given by the deltas and timestamp

    Parameters
    ----------
    motor: integer
        Strut ID for the motorCurrent to query
    delta_start: integer
        Time in seconds before the timestamp
    delta_end: integer
        Time in seconds after the timestamp
    timestamp: pd.dataframe index
        Time when error occurred
    '''
    
    # Convert index strings to datetime object and then astropytime
    timestamp = Time(pd.to_datetime(timestamp), scale='utc')

    delta1 = TimeDelta(delta_start, format='sec')
    delta2 = TimeDelta(delta_end, format='sec')
    
    start_time = (timestamp - delta1).to_value('isot', subfmt='date_hms') + 'Z'
    end_time = (timestamp + delta2).to_value('isot', subfmt='date_hms') + 'Z'

    query = f'''
            SELECT stddev("motorCurrent{motor}") 
            AS "stddev_motorCurrent{motor}"
            FROM "efd"."autogen"."lsst.sal.MTHexapod.electrical" 
            WHERE time > '{start_time}' 
            AND time < '{end_time}'
            GROUP BY time(100ms) FILL(null)
            '''

    return(query)

In [ ]:
median_val = {}
for motor in range(N_STRUTS):
    median_val[motor] = []

timestamp_list = []

# Cycle through timestamps and motors, then append each to their respective list
for timestamp in timestamps_error_code:
    timestamp_list.append(timestamp)
    
    for motor in range(N_STRUTS):
        df_currentmotor = await efd_client.influx_client.query(
        fault_current_noise_query(motor, delta_start, delta_end, timestamp)
        )
        noisemedian = df_currentmotor[f'stddev_motorCurrent{motor}'].median()
        median_val[motor].append(noisemedian)
        if verbose == True: 
            print(f'The median of noise for strut id {motor} on timestamp {timestamp} is:{noisemedian}')

In [ ]:
median_df = {}

for motor in range(N_STRUTS):
    df = pd.DataFrame({
        'Time': timestamp_list,
        f'Median Noise {motor}': median_val[motor]
    })
    
    df["Time"] = pd.to_datetime(df["Time"])
    df.set_index("Time", inplace=True)
    median_df[motor] = df
    if verbose == True:
        print(median_df[motor])

In [ ]:
# We can finally plot them

ax = median_df[0].plot.area(figsize=(15,5), alpha=0.5, color='cyan')
median_df[1].plot.area(ax=ax, alpha=0.5, color='blue' )
median_df[2].plot.area(ax=ax, alpha=0.5, color='violet')
median_df[3].plot.area(ax=ax, alpha=0.5, color='purple')
median_df[4].plot.area(ax=ax, alpha=0.5, color='red')
median_df[5].plot.area(ax=ax, alpha=0.5, color='orange')

plt.ylim(0,3)
plt.ylabel('Median of Noise current')
plt.xlabel('Error timestamps')

plt.grid(True)
#plt.savefig('Median_stddev_struts.png', dpi=400)
plt.show()

With this plot we can visually see that the possible main problem is the Strut 6, as the Current Noise is usually higher prior to a fault.

In [ ]:
# I will consider a threshold of 0.5 (UNITS?)
threshold = 0.5

for motor in range(N_STRUTS):
    df = median_df[motor]
    df = df[df[f'Median Noise {motor}'] > threshold].index
    plt.hist(df, bins=15, alpha=0.5, label=f'Motor {motor}')

plt.xlabel("Time")
plt.ylabel("Frequency")
plt.xticks(rotation=20)
plt.title("Times when median noise > 0.5")
plt.legend()
plt.grid(True)
plt.show()

## Oscillations of the current

To understand how the current behaves we have to see the components of the frecuency.

We're going back to one single timestamp

Not yet commented

In [ ]:
timestamp = '2025-07-28 20:17:20'
df_currentmotor = {}

for motor in range(N_STRUTS):
    df_currentmotor = await efd_client.influx_client.query(
        fault_current_noise_query(motor, delta_start, delta_end, timestamp)
    )
    
    signal = df_currentmotor.values.flatten()
    time = df_currentmotor.index
    signal = detrend(signal)
    dt = (time[1] - time[0]).total_seconds()

    fft_vals = np.fft.rfft(signal)
    freqs = np.fft.rfftfreq(len(signal), dt)
    power = np.abs(fft_vals)**2
    
    if verbose == True: 
        df_currentmotor.plot(figsize=(15,5), alpha=0.5, color='blue')
        plt.ylim(0,3.5)
        plt.ylabel('Median of Noise current')
        plt.xlabel('Error timestamps')
        plt.title(f'Current Noise for motor {motor}')
        plt.grid(True)
        plt.show()
        plt.clf()
        
        plt.figure(figsize=(15,5))
        plt.plot(freqs, power, color='blue')
        plt.xlabel('Frequency [Hz]')
        plt.ylabel('Power')
        plt.xticks(np.arange(0, 5, 0.25))
        plt.title(f'FFT Power Spectrum of Motor {motor} Noise Current')
        plt.grid(True)
        plt.show()
        plt.clf()
    